# 🔮 Markov Chain Signal Generator + Backtester — IHSG Universe

**End-to-end pipeline:** GMM-based regime detection → walk-forward signal generation → return simulation → benchmark comparison.

---
| Cell | Purpose |
|---|---|
| 1 | Install dependencies |
| 2 | Core functions (HMM engine + display helpers) |
| 3 | **⭐ Backtest config & run** — edit ticker / dates here |
| 3B | Multi-stock comparison overlay |
| 4 | Single-stock live signal (optional) |
| 5 | Universe batch scanner (optional) |

> **Ticker format:** bare IDX code — `BBCA`, `TLKM`, `GOTO`. `.JK` appended automatically.

In [ ]:
# ── Cell 1 · Install dependencies ────────────────────────────────────────────
!pip install -q yfinance scikit-learn plotly kaleido
print("✅ All dependencies ready.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.0/69.0 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.3/49.3 kB 3.1 MB/s eta 0:00:00
✅ All dependencies ready.


In [ ]:
# ── Cell 2 · Core Engine ──────────────────────────────────────────────────────
import pandas as pd
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from datetime import datetime, timedelta
import yfinance as yf
from sklearn.mixture import GaussianMixture
from sklearn.preprocessing import StandardScaler
from IPython.display import display, HTML
import warnings
warnings.filterwarnings("ignore")

# ── Popular IHSG tickers ──────────────────────────────────────────────────────
IHSG_UNIVERSE = {
    "BBCA": "Bank Central Asia",      "BBRI": "Bank Rakyat Indonesia",
    "BMRI": "Bank Mandiri",           "BBNI": "Bank Negara Indonesia",
    "BRIS": "Bank Syariah Indonesia", "TLKM": "Telkom Indonesia",
    "EXCL": "XL Axiata",              "ISAT": "Indosat",
    "UNVR": "Unilever Indonesia",     "ICBP": "Indofood CBP",
    "INDF": "Indofood",               "MYOR": "Mayora Indah",
    "ADRO": "Adaro Energy",           "PTBA": "Bukit Asam",
    "ITMG": "Indo Tambangraya",       "PGAS": "Perusahaan Gas Negara",
    "ASII": "Astra International",    "KLBF": "Kalbe Farma",
    "SIDO": "Sido Muncul",            "BSDE": "Bumi Serpong Damai",
    "SMRA": "Summarecon Agung",       "SMGR": "Semen Indonesia",
    "INTP": "Indocement",             "GOTO": "GoTo Gojek Tokopedia",
    "BUKA": "Bukalapak",              "ACES": "Ace Hardware Indonesia",
    "^JKSE": "IHSG Index",
}


# ── Utilities ─────────────────────────────────────────────────────────────────
def to_jk(symbol):
    s = symbol.strip().upper()
    if s.startswith("^") or s.endswith(".JK"):
        return s
    return s + ".JK"


def fetch_data(symbol, start, end):
    df = yf.download(to_jk(symbol), start=start, end=end, progress=False)
    if isinstance(df.columns, pd.MultiIndex):
        df.columns = df.columns.get_level_values(0)
    return df


def show_html(html):
    display(HTML(html))


# ── Feature engineering ───────────────────────────────────────────────────────
def calc_rsi(prices, period=14):
    out = [50.0] * len(prices)
    for i in range(period, len(prices)):
        ch = [prices[j] - prices[j - 1] for j in range(i - period + 1, i + 1)]
        g = sum(c for c in ch if c > 0) / period
        lo = sum(-c for c in ch if c < 0) / period
        out[i] = 100 if lo == 0 else 100 - 100 / (1 + g / lo)
    return out


def build_features(closes, volumes):
    n = len(closes)
    ret = [0.0] + [(closes[i] - closes[i - 1]) / closes[i - 1] for i in range(1, n)]
    vols, vrat = [], []
    mom10 = [0.0] * 10 + [(closes[i] - closes[i - 10]) / closes[i - 10] for i in range(10, n)]
    mom5 = [0.0] * 5 + [(closes[i] - closes[i - 5]) / closes[i - 5] for i in range(5, n)]
    rsi = calc_rsi(closes)
    for i in range(n):
        if i < 20:
            vols.append(0.02)
            vrat.append(1.0)
        else:
            chunk = ret[i - 19: i + 1]
            mu = sum(chunk) / 20
            vols.append((sum((r - mu) ** 2 for r in chunk) / 20) ** 0.5)
            avol = sum(volumes[i - 19: i + 1]) / 20
            vrat.append(volumes[i] / avol if avol else 1.0)
    feats = [[ret[i], vols[i], vrat[i], mom10[i], mom5[i], rsi[i] / 100] for i in range(n)]
    return np.array(feats), ret


# ── HMM fit ───────────────────────────────────────────────────────────────────
def fit_hmm(scaled, n_components=3):
    m = GaussianMixture(
        n_components=n_components, covariance_type="diag",
        random_state=42, max_iter=150, n_init=5,
    )
    m.fit(scaled)
    return m


def remap_states(states, returns, n_regimes):
    means = {
        r: np.mean([returns[i] for i in range(len(returns)) if states[i] == r])
        for r in range(n_regimes)
    }
    order = sorted(means, key=means.get)  # Bear=0, Sideways=1, Bull=2
    rmap = {old: new for new, old in enumerate(order)}
    return [rmap[s] for s in states], rmap


# ── Live signal analysis ──────────────────────────────────────────────────────
def analyze_stock(symbol, lookback_days=300, n_regimes=3):
    end_dt = datetime.now()
    start_dt = end_dt - timedelta(days=int(lookback_days * 1.6))
    data = fetch_data(symbol, start_dt, end_dt)
    if data.empty or len(data) < 80:
        raise ValueError(f"Insufficient data for {to_jk(symbol)}")

    closes = data["Close"].tolist()
    volumes = data["Volume"].tolist()
    raw_feats, returns = build_features(closes, volumes)

    burn = 30
    feats = raw_feats[burn:]
    returns = returns[burn:]
    dates = data.index[burn:]

    scaler = StandardScaler()
    scaled = scaler.fit_transform(feats)
    model = fit_hmm(scaled, n_regimes)
    states = model.predict(scaled)
    probs = model.predict_proba(scaled)
    mapped, _ = remap_states(states, returns, n_regimes)

    names = {0: "Bear 📉", 1: "Sideways ➡️", 2: "Bull 📈"}
    clrs = {0: "#e74c3c", 1: "#f39c12", 2: "#27ae60"}

    rstats = {}
    for r in range(n_regimes):
        sub = [returns[i] for i in range(len(returns)) if mapped[i] == r]
        if not sub:
            continue
        pd_days = sum(1 for i in range(len(mapped) - 1) if mapped[i] == r)
        persist = sum(1 for i in range(1, len(mapped)) if mapped[i - 1] == r and mapped[i] == r)
        rstats[r] = dict(
            name=names[r], color=clrs[r],
            days=len(sub), pct=len(sub) / len(mapped) * 100,
            avg_return=np.mean(sub) * 100, volatility=np.std(sub) * 100,
            persistence=(persist / pd_days * 100) if pd_days else 0,
            sharpe=(np.mean(sub) / np.std(sub) * 252 ** 0.5) if np.std(sub) else 0,
        )

    cur_r = mapped[-1]
    conf = float(probs[-1].max())
    if conf >= 0.65:
        sig = "BUY" if cur_r == 2 else ("SELL" if cur_r == 0 else "HOLD")
    else:
        sig = "HOLD"
    sig_clr = {"BUY": "#27ae60", "SELL": "#e74c3c", "HOLD": "#f39c12"}[sig]

    return dict(
        ticker=to_jk(symbol), symbol=symbol.upper(),
        name=IHSG_UNIVERSE.get(symbol.upper(), ""),
        signal=sig, sig_color=sig_clr,
        strength=min(10, max(1, int(conf * 12))),
        regime=names[cur_r], regime_id=cur_r, confidence=conf * 100,
        regime_stats=rstats, current_stats=rstats.get(cur_r, {}),
        data=data, dates=dates, states=mapped, returns=returns, probs=probs,
        n_regimes=n_regimes, names=names, colors_regime=clrs,
    )


# ── Display helpers ───────────────────────────────────────────────────────────
def show_signal_card(r):
    bar = "█" * r["strength"] + "░" * (10 - r["strength"])
    bg = {"BUY": "#d5f5e3", "SELL": "#fadbd8", "HOLD": "#fef9e7"}[r["signal"]]
    nm = f"{r['symbol']} – {r['name']}" if r["name"] else r["symbol"]
    show_html(f"""
    <div style="background:{bg};border:3px solid {r['sig_color']};border-radius:12px;
                padding:20px;margin:12px 0;font-family:sans-serif">
      <h2 style="color:{r['sig_color']};margin:0">{r['signal']} — {nm}</h2>
      <h3 style="margin:4px 0;color:#555">{r['regime']}</h3>
      <p>Confidence: <strong>{r['confidence']:.1f}%</strong></p>
      <p style="font-size:1.3rem;letter-spacing:1px">{bar} {r['strength']}/10</p>
    </div>""")


def show_regime_table(r):
    rows = "".join(
        f"<tr style='background:{s['color']}22'>"
        f"<td style='padding:8px'>{s['name']}</td>"
        f"<td style='padding:8px'>{s['days']}</td>"
        f"<td style='padding:8px'>{s['pct']:.1f}%</td>"
        f"<td style='padding:8px'>{s['avg_return']:.3f}%</td>"
        f"<td style='padding:8px'>{s['volatility']:.2f}%</td>"
        f"<td style='padding:8px'>{s['persistence']:.1f}%</td>"
        f"<td style='padding:8px'>{s['sharpe']:.2f}</td></tr>"
        for s in r["regime_stats"].values()
    )
    show_html(f"""<table style='border-collapse:collapse;width:100%;font-family:sans-serif'>
      <thead style='background:#2c3e50;color:#fff'><tr>
        <th style='padding:10px'>Regime</th><th>Days</th><th>%</th>
        <th>Avg Ret/day</th><th>Volatility</th><th>Persistence</th><th>Ann.Sharpe</th>
      </tr></thead><tbody>{rows}</tbody></table>""")


print("✅ Core engine loaded. Proceed to Cell 3.")

✅ Core engine loaded. Proceed to Cell 3.


---
## ⭐ Cell 3 — Backtest Config & Run

Edit the CONFIG block, then **Run Cell 3**.

| Variable | What it controls |
|---|---|
| `SYMBOL` | IDX ticker — `BBCA`, `TLKM`, `GOTO`, etc. |
| `START_DATE` | Backtest start `YYYY-MM-DD` |
| `END_DATE` | Backtest end — `"today"` for current date |
| `TRAIN_WINDOW` | Rolling bars used to train HMM at each step |
| `N_REGIMES` | Number of regimes (3 = Bear / Sideways / Bull) |
| `CONFIDENCE_THRESHOLD` | Min confidence for a directional position |
| `STOP_LOSS_PCT` | Exit long if price falls this % from entry |
| `TRANSACTION_COST` | One-way cost per trade (0.0015 = 0.15%) |
| `INITIAL_CAPITAL` | Starting capital in Rupiah |
| `ALLOW_SHORT` | `True` to go short in Bear regime |

In [31]:
# ══════════════════════════════════════════════════════════════════════════════
#  ⚙️  BACKTEST CONFIG  — edit anything below this line
# ══════════════════════════════════════════════════════════════════════════════

SYMBOL               = "AKRA"        # IDX ticker (no .JK needed)
START_DATE           = "2025-01-01"  # backtest start
END_DATE             = "today"       # "today" or "YYYY-MM-DD"

N_REGIMES            = 3             # 3 = Bear / Sideways / Bull
TRAIN_WINDOW         = 120           # rolling bars to train HMM
CONFIDENCE_THRESHOLD = 0.65          # 0–1; higher = fewer but stronger signals

STOP_LOSS_PCT        = 0.05          # 5% stop-loss from entry price
TRANSACTION_COST     = 0.0015        # 0.15% per trade (typical IDX cost)
ALLOW_SHORT          = False         # True to short in Bear regime

INITIAL_CAPITAL      = 100_000_000   # Rp 100 juta

# ══════════════════════════════════════════════════════════════════════════════
#  Everything below runs automatically — no edits needed
# ══════════════════════════════════════════════════════════════════════════════

import pandas as pd
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from datetime import timedelta
import yfinance as yf
from sklearn.mixture import GaussianMixture
from sklearn.preprocessing import StandardScaler
from IPython.display import display, HTML
import warnings
warnings.filterwarnings("ignore")


# ── Core backtest engine ──────────────────────────────────────────────────────
def run_backtest(symbol, start_date, end_date,
                 n_regimes=3, train_window=120,
                 confidence_threshold=0.65,
                 transaction_cost=0.001,
                 stop_loss_pct=0.07,
                 initial_capital=100_000_000,
                 allow_short=False):
    """
    Walk-forward HMM backtest.
    At each bar the model is trained on the previous `train_window` bars,
    the regime is predicted, and a position is taken:
      Bull  -> Long  (if confidence >= threshold)
      Bear  -> Short (only if allow_short=True) or Cash
      Sideways / low confidence -> Cash
    """
    dt_start  = pd.to_datetime(start_date)
    dt_end    = pd.to_datetime(end_date)
    buf_start = dt_start - timedelta(days=int(train_window * 2))

    print(f"Downloading {to_jk(symbol)} and ^JKSE ...")
    stock = fetch_data(symbol, buf_start, dt_end)
    ihsg  = fetch_data("^JKSE", buf_start, dt_end)

    if stock.empty or len(stock) < train_window + 10:
        raise ValueError(f"Not enough data for {symbol}. Try a wider date range.")

    common = stock.index.intersection(ihsg.index)
    stock = stock.loc[common]
    ihsg  = ihsg.loc[common]

    closes_all  = stock["Close"].tolist()
    volumes_all = stock["Volume"].tolist()
    dates_all   = list(stock.index)
    ihsg_closes = ihsg["Close"].tolist()

    print(f"Running walk-forward HMM ({train_window}-day window) on {len(dates_all)} bars ...")
    signals_all, conf_all, regime_all = [], [], []

    for i in range(len(dates_all)):
        if i < train_window:
            signals_all.append(0); conf_all.append(0.0); regime_all.append(1)
            continue

        w_closes  = closes_all[i - train_window: i + 1]
        w_volumes = volumes_all[i - train_window: i + 1]
        feats, rets = build_features(w_closes, w_volumes)
        burn = 20
        feats = feats[burn:]; rets = rets[burn:]

        scaler = StandardScaler()
        scaled = scaler.fit_transform(feats)
        try:
            model   = fit_hmm(scaled, n_regimes)
            states  = model.predict(scaled)
            probs   = model.predict_proba(scaled)
            mapped, _ = remap_states(states, rets, n_regimes)
        except Exception:
            signals_all.append(0); conf_all.append(0.0); regime_all.append(1)
            continue

        cur_r = mapped[-1]
        conf  = float(probs[-1].max())
        if conf >= confidence_threshold:
            if cur_r == 2:   pos = 1
            elif cur_r == 0: pos = -1 if allow_short else 0
            else:            pos = 0
        else:
            pos = 0

        signals_all.append(pos); conf_all.append(conf); regime_all.append(cur_r)

    # ── Trim to backtest window ───────────────────────────────────────────────
    bt_idx      = [i for i, d in enumerate(dates_all) if d >= dt_start]
    bt_dates    = [dates_all[i]   for i in bt_idx]
    bt_closes   = [closes_all[i]  for i in bt_idx]
    bt_ihsg     = [ihsg_closes[i] for i in bt_idx]
    bt_signals  = [signals_all[i] for i in bt_idx]
    bt_conf     = [conf_all[i]    for i in bt_idx]
    bt_regimes  = [regime_all[i]  for i in bt_idx]

    bt_ret = [0.0]
    for i in range(1, len(bt_closes)):
        bt_ret.append((bt_closes[i] - bt_closes[i - 1]) / bt_closes[i - 1])

    # ── Simulate equity ───────────────────────────────────────────────────────
    equity     = [initial_capital]
    position   = 0
    entry_px   = None
    trade_log  = []
    n_trades   = 0
    total_cost = 0.0

    for i in range(1, len(bt_dates)):
        new_pos = bt_signals[i]
        prev_eq = equity[-1]

        # Stop-loss
        if position == 1 and entry_px:
            if (bt_closes[i] - entry_px) / entry_px <= -stop_loss_pct:
                new_pos = 0

        cost = 0.0
        if new_pos != position:
            cost = prev_eq * transaction_cost
            total_cost += cost
            n_trades   += 1
            action = ("BUY" if new_pos == 1 else ("SHORT" if new_pos == -1 else "EXIT"))
            trade_log.append({"date": bt_dates[i], "action": action,
                              "price": bt_closes[i], "conf": bt_conf[i]})
            entry_px = bt_closes[i] if new_pos != 0 else None
            position = new_pos

        daily_ret = bt_ret[i]
        if position == 1:
            pnl = (prev_eq - cost) * daily_ret
        elif position == -1:
            pnl = (prev_eq - cost) * (-daily_ret)
        else:
            pnl = 0.0
        equity.append(prev_eq + pnl - cost)

    # ── Benchmarks ────────────────────────────────────────────────────────────
    stock_bah = [initial_capital]
    for i in range(1, len(bt_closes)):
        stock_bah.append(stock_bah[-1] * (1 + bt_ret[i]))

    ihsg_ret = [0.0]
    for i in range(1, len(bt_ihsg)):
        ihsg_ret.append((bt_ihsg[i] - bt_ihsg[i - 1]) / bt_ihsg[i - 1])

    ihsg_bah = [initial_capital]
    for i in range(1, len(bt_dates)):
        ihsg_bah.append(ihsg_bah[-1] * (1 + ihsg_ret[i]))

    # ── Metrics ───────────────────────────────────────────────────────────────
    def calc_metrics(curve, daily_rets):
        arr = np.array(daily_rets)
        total_ret = (curve[-1] - curve[0]) / curve[0] * 100
        n_years   = len(curve) / 252
        cagr      = ((curve[-1] / curve[0]) ** (1 / n_years) - 1) * 100 if n_years > 0 else 0
        sharpe    = (arr.mean() / arr.std() * 252 ** 0.5) if arr.std() else 0
        peak = curve[0]; mdd = 0.0
        for v in curve:
            if v > peak: peak = v
            dd = (v - peak) / peak
            if dd < mdd: mdd = dd
        win_rate = sum(1 for r in arr if r > 0) / len(arr) * 100 if len(arr) else 0
        return dict(total_ret=total_ret, cagr=cagr, sharpe=sharpe,
                    max_dd=mdd * 100, win_rate=win_rate)

    strat_daily = [(equity[i] - equity[i - 1]) / equity[i - 1]
                   if equity[i - 1] else 0
                   for i in range(1, len(equity))]

    print(f"Done. {n_trades} trades executed.")
    return dict(
        symbol=symbol.upper(), start=bt_dates[0], end=bt_dates[-1],
        dates=bt_dates, equity=equity, stock_bah=stock_bah, ihsg_bah=ihsg_bah,
        bt_closes=bt_closes, bt_signals=bt_signals, bt_regimes=bt_regimes,
        bt_conf=bt_conf, strat_daily=strat_daily,
        m_strat=calc_metrics(equity,     strat_daily),
        m_stock=calc_metrics(stock_bah,  bt_ret[1:]),
        m_ihsg =calc_metrics(ihsg_bah,   ihsg_ret[1:]),
        n_trades=n_trades, total_cost=total_cost,
        trade_log=pd.DataFrame(trade_log),
        initial_capital=initial_capital,
        train_window=train_window, allow_short=allow_short,
        n_regimes=n_regimes,
        confidence_threshold=confidence_threshold,
        stop_loss_pct=stop_loss_pct,
        transaction_cost=transaction_cost,
    )


# ── Metric display ────────────────────────────────────────────────────────────
def show_metrics(bt):
    ms, mk, mi = bt["m_strat"], bt["m_stock"], bt["m_ihsg"]
    def fr(v): c="#27ae60" if v>=0 else "#e74c3c"; return f"<span style='color:{c};font-weight:bold'>{v:+.2f}%</span>"
    def fd(v): c="#e74c3c" if v<-1 else "#888"; return f"<span style='color:{c}'>{v:.2f}%</span>"
    rows_data = [
        ("Total Return",       fr(ms["total_ret"]),  fr(mk["total_ret"]),  fr(mi["total_ret"])),
        ("CAGR",               fr(ms["cagr"]),        fr(mk["cagr"]),        fr(mi["cagr"])),
        ("Sharpe Ratio",       f"{ms['sharpe']:.2f}", f"{mk['sharpe']:.2f}", f"{mi['sharpe']:.2f}"),
        ("Max Drawdown",       fd(ms["max_dd"]),      fd(mk["max_dd"]),      fd(mi["max_dd"])),
        ("Win Rate (days)",    f"{ms['win_rate']:.1f}%", f"{mk['win_rate']:.1f}%", f"{mi['win_rate']:.1f}%" ),
        ("Final Value",        f"Rp {bt['equity'][-1]:,.0f}", f"Rp {bt['stock_bah'][-1]:,.0f}", f"Rp {bt['ihsg_bah'][-1]:,.0f}"),
        ("Trades",             str(bt["n_trades"]),   "1 (B&H)",             "1 (B&H)"),
        ("Transaction Costs",  f"Rp {bt['total_cost']:,.0f}", "—", "—"),
    ]
    trs = "".join(
        f"<tr style='border-bottom:1px solid #eee'>"
        f"<td style='padding:9px 14px;color:#555;font-weight:500'>{lb}</td>"
        f"<td style='padding:9px 14px;text-align:center'>{a}</td>"
        f"<td style='padding:9px 14px;text-align:center;color:#3498db'>{b}</td>"
        f"<td style='padding:9px 14px;text-align:center;color:#e67e22'>{c}</td></tr>"
        for lb, a, b, c in rows_data
    )
    cfg = (f"Train window: {bt['train_window']}d · "
           f"Confidence ≥{bt['confidence_threshold']*100:.0f}% · "
           f"Stop-loss: {bt['stop_loss_pct']*100:.0f}% · "
           f"Short: {'✅' if bt['allow_short'] else '❌'} · "
           f"Cost: {bt['transaction_cost']*100:.2f}%/trade")
    display(HTML(f"""
    <div style='font-family:sans-serif;max-width:820px'>
      <h3 style='margin-bottom:4px'>Performance Summary — {bt['symbol']}</h3>
      <p style='color:#888;font-size:.85rem;margin-bottom:12px'>{cfg}</p>
      <table style='width:100%;border-collapse:collapse;background:#fff;
                    box-shadow:0 2px 8px rgba(0,0,0,.08);border-radius:10px;overflow:hidden'>
        <thead style='background:#2c3e50;color:#fff'><tr>
          <th style='padding:12px 14px;text-align:left'>Metric</th>
          <th style='padding:12px 14px'>🤖 HMM Strategy</th>
          <th style='padding:12px 14px;color:#85c1e9'>{bt['symbol']} Buy &amp; Hold</th>
          <th style='padding:12px 14px;color:#f0b27a'>IHSG Buy &amp; Hold</th>
        </tr></thead>
        <tbody>{trs}</tbody>
      </table></div>"""))


# ── Main chart (4 panels) ─────────────────────────────────────────────────────
def plot_backtest(bt):
    dates = bt["dates"]; equity = bt["equity"]
    stock_bah = bt["stock_bah"]; ihsg_bah = bt["ihsg_bah"]
    closes = bt["bt_closes"]; sigs = bt["bt_signals"]; regs = bt["bt_regimes"]
    base = bt["initial_capital"]
    eq_n   = [v / base * 100 for v in equity]
    stk_n  = [v / base * 100 for v in stock_bah]
    ihsg_n = [v / base * 100 for v in ihsg_bah]

    def dd_series(curve):
        peak = curve[0]; out = []
        for v in curve:
            if v > peak: peak = v
            out.append((v - peak) / peak * 100)
        return out

    fig = make_subplots(
        rows=4, cols=1, shared_xaxes=True,
        subplot_titles=[
            "Normalised Return (Base = 100)",
            "Drawdown (%)",
            f"{bt['symbol']} Price + Position (green=long, red=short)",
            "Regime State",
        ],
        row_heights=[0.38, 0.18, 0.25, 0.19],
        vertical_spacing=0.045,
    )

    # Panel 1 — equity curves
    fig.add_trace(go.Scatter(x=dates, y=eq_n,   name="HMM Strategy",
                             line=dict(color="#2ecc71", width=2.2)), row=1, col=1)
    fig.add_trace(go.Scatter(x=dates, y=stk_n,  name=f"{bt['symbol']} B&H",
                             line=dict(color="#3498db", width=1.8, dash="dot")), row=1, col=1)
    fig.add_trace(go.Scatter(x=dates, y=ihsg_n, name="IHSG B&H",
                             line=dict(color="#e67e22", width=1.8, dash="dash")), row=1, col=1)
    fig.add_hline(y=100, line_dash="dot", line_color="#95a5a6", line_width=1, row=1, col=1)

    # Panel 2 — drawdown
    fig.add_trace(go.Scatter(x=dates, y=dd_series(equity),    name="Strategy DD",
                             fill="tozeroy", fillcolor="rgba(46,204,113,0.18)",
                             line=dict(color="#27ae60", width=1.5)), row=2, col=1)
    fig.add_trace(go.Scatter(x=dates, y=dd_series(stock_bah), name=f"{bt['symbol']} DD",
                             line=dict(color="#3498db", width=1, dash="dot")), row=2, col=1)
    fig.add_trace(go.Scatter(x=dates, y=dd_series(ihsg_bah),  name="IHSG DD",
                             line=dict(color="#e67e22", width=1, dash="dash")), row=2, col=1)

    # Panel 3 — price + shading
    fig.add_trace(go.Scatter(x=dates, y=closes, name="Price",
                             line=dict(color="#2c3e50", width=1.5)), row=3, col=1)
    sig_colors = {1: "rgba(46,204,113,0.22)", -1: "rgba(231,76,60,0.22)", 0: "rgba(0,0,0,0)"}
    prev_s = sigs[0]; prev_d = dates[0]
    for i in range(1, len(dates)):
        if sigs[i] != prev_s or i == len(dates) - 1:
            if prev_s != 0:
                fig.add_vrect(x0=prev_d, x1=dates[i],
                              fillcolor=sig_colors[prev_s], layer="below",
                              line_width=0, row=3, col=1)
            prev_s = sigs[i]; prev_d = dates[i]

    tl = bt["trade_log"]
    if not tl.empty:
        buys  = tl[tl["action"] == "BUY"]
        exits = tl[tl["action"] == "EXIT"]
        if not buys.empty:
            fig.add_trace(go.Scatter(x=buys["date"], y=buys["price"], mode="markers",
                                     name="Entry", marker=dict(symbol="triangle-up", size=9,
                                     color="#2ecc71", line=dict(color="white", width=1))), row=3, col=1)
        if not exits.empty:
            fig.add_trace(go.Scatter(x=exits["date"], y=exits["price"], mode="markers",
                                     name="Exit",  marker=dict(symbol="triangle-down", size=9,
                                     color="#e74c3c", line=dict(color="white", width=1))), row=3, col=1)

    # Panel 4 — regime bars
    rnames  = {0: "Bear", 1: "Sideways", 2: "Bull"}
    rcolors = {0: "#e74c3c", 1: "#f39c12", 2: "#27ae60"}
    for r in [2, 1, 0]:
        fig.add_trace(go.Bar(x=dates, y=[1 if regs[i] == r else 0 for i in range(len(dates))],
                             name=rnames[r], marker_color=rcolors[r]), row=4, col=1)

    start_str = bt["start"].strftime("%d %b %Y")
    end_str   = bt["end"].strftime("%d %b %Y")
    fig.update_layout(
        title=dict(text=f"🔮 {bt['symbol']} · HMM Backtest — {start_str} to {end_str}",
                   font=dict(size=16)),
        height=940, template="plotly_white", barmode="stack",
        legend=dict(orientation="h", y=1.03, x=0, font=dict(size=11)),
        hovermode="x unified", margin=dict(t=100, b=40),
    )
    fig.update_yaxes(title_text="Index", row=1, col=1)
    fig.update_yaxes(title_text="DD %",  row=2, col=1)
    fig.update_yaxes(title_text="Rp",    row=3, col=1)
    fig.show()


# ── Monthly heatmap ───────────────────────────────────────────────────────────
def plot_monthly_heatmap(bt):
    s = pd.Series(bt["equity"], index=pd.DatetimeIndex(bt["dates"]))
    m = s.resample("ME").last().pct_change().dropna() * 100
    m = m.reset_index(); m.columns = ["date", "ret"]
    m["Year"] = m["date"].dt.year; m["Month"] = m["date"].dt.month
    pivot = m.pivot(index="Year", columns="Month", values="ret")
    mn = ["Jan","Feb","Mar","Apr","May","Jun","Jul","Aug","Sep","Oct","Nov","Dec"]
    pivot.columns = [mn[c - 1] for c in pivot.columns]
    fig = go.Figure(go.Heatmap(
        z=pivot.values, x=pivot.columns.tolist(),
        y=[str(y) for y in pivot.index],
        colorscale="RdYlGn", zmid=0,
        text=[[f"{v:.1f}%" if not np.isnan(v) else "" for v in row] for row in pivot.values],
        texttemplate="%{text}", textfont=dict(size=11),
        hovertemplate="%{y} %{x}: %{z:.2f}%<extra></extra>",
        colorbar=dict(title="Ret %"),
    ))
    fig.update_layout(
        title=f"📅 {bt['symbol']} Strategy — Monthly Return Heatmap",
        height=max(280, 60 * len(pivot) + 110), template="plotly_white",
    )
    fig.show()


# ── Trade log ─────────────────────────────────────────────────────────────────
def show_trade_log(bt, n=25):
    tl = bt["trade_log"]
    if tl.empty:
        print("No trades executed.")
        return
    t2 = tl.copy()
    t2["date"]  = pd.DatetimeIndex(t2["date"]).strftime("%d %b %Y")
    t2["price"] = t2["price"].apply(lambda v: f"Rp {v:,.0f}")
    t2["conf"]  = t2["conf"].apply(lambda v: f"{v*100:.1f}%")
    t2.columns  = ["Date", "Action", "Price", "Confidence"]
    print(f"Trade Log (last {min(n, len(t2))} trades):")
    display(t2.tail(n).reset_index(drop=True).style.applymap(
        lambda v: ("background-color:#d5f5e3" if v == "BUY" else
                   "background-color:#fadbd8" if v == "EXIT" else ""),
        subset=["Action"]
    ))


# ── Run ───────────────────────────────────────────────────────────────────────
end_val = str(pd.Timestamp.today().date()) if END_DATE == "today" else END_DATE

bt_result = run_backtest(
    symbol=SYMBOL, start_date=START_DATE, end_date=end_val,
    n_regimes=N_REGIMES, train_window=TRAIN_WINDOW,
    confidence_threshold=CONFIDENCE_THRESHOLD,
    transaction_cost=TRANSACTION_COST, stop_loss_pct=STOP_LOSS_PCT,
    initial_capital=INITIAL_CAPITAL, allow_short=ALLOW_SHORT,
)
show_metrics(bt_result)
plot_backtest(bt_result)
plot_monthly_heatmap(bt_result)
show_trade_log(bt_result)

Running walk-forward HMM (120-day window) on 429 bars ...
Done. 32 trades executed.


Metric,🤖 HMM Strategy,AKRA Buy & Hold,IHSG Buy & Hold
Total Return,+139.20%,+23.13%,+15.51%
CAGR,+127.06%,+21.61%,+14.52%
Sharpe Ratio,2.75,0.65,0.76
Max Drawdown,-11.04%,-31.30%,-17.76%
Win Rate (days),14.6%,45.3%,56.2%
Final Value,"Rp 239,195,194","Rp 123,125,230","Rp 115,508,086"
Trades,32,1 (B&H),1 (B&H)
Transaction Costs,"Rp 8,221,061",—,—


Trade Log (last 25 trades):


,Date,Action,Price,Confidence
0,11 Apr 2025,EXIT,Rp 971,100.0%
1,14 Apr 2025,BUY,Rp 994,100.0%
2,08 May 2025,EXIT,"Rp 1,193",100.0%
3,09 May 2025,BUY,"Rp 1,198",100.0%
4,14 May 2025,EXIT,"Rp 1,207",100.0%
5,21 May 2025,BUY,"Rp 1,323",100.0%
6,22 May 2025,EXIT,"Rp 1,289",90.8%
7,21 Oct 2025,BUY,"Rp 1,130",98.4%
8,22 Oct 2025,EXIT,"Rp 1,095",99.9%
9,24 Oct 2025,BUY,"Rp 1,210",100.0%


---
## Cell 3B — Multi-Stock Comparison

Compare the HMM strategy across several tickers on the same chart vs IHSG.

In [ ]:
# ── Cell 3B · Multi-stock backtest comparison ─────────────────────────────────

COMPARE_SYMBOLS = ["BBCA", "BBRI", "TLKM", "ASII", "ADRO"]
COMPARE_START   = "2021-01-01"
COMPARE_END     = "today"    # "today" or "YYYY-MM-DD"

_kw = dict(n_regimes=3, train_window=120, confidence_threshold=0.65,
           transaction_cost=0.0015, stop_loss_pct=0.07,
           initial_capital=100_000_000, allow_short=False)
_end = str(pd.Timestamp.today().date()) if COMPARE_END == "today" else COMPARE_END

summary_rows = []
all_curves   = {}
ihsg_curve   = None
all_dates    = None

for sym in COMPARE_SYMBOLS:
    try:
        print(f"\n=== {sym} ===")
        bt = run_backtest(sym, COMPARE_START, _end, **_kw)
        m  = bt["m_strat"]
        summary_rows.append({
            "Symbol":     sym,
            "Total Ret":  f"{m['total_ret']:+.2f}%",
            "CAGR":       f"{m['cagr']:+.2f}%",
            "Sharpe":     f"{m['sharpe']:.2f}",
            "Max DD":     f"{m['max_dd']:.2f}%",
            "Win Rate":   f"{m['win_rate']:.1f}%",
            "Trades":     bt["n_trades"],
        })
        base = bt["initial_capital"]
        all_curves[sym] = [v / base * 100 for v in bt["equity"]]
        all_dates = bt["dates"]
        if ihsg_curve is None:
            ihsg_curve = [v / base * 100 for v in bt["ihsg_bah"]]
    except Exception as e:
        print(f"  ERROR {sym}: {e}")

print("\n===== Comparison Summary =====")
display(pd.DataFrame(summary_rows))

if all_curves and all_dates:
    PALETTE = ["#2ecc71","#3498db","#9b59b6","#e67e22","#1abc9c","#e74c3c"]
    fig_cmp = go.Figure()
    for i, (sym, curve) in enumerate(all_curves.items()):
        fig_cmp.add_trace(go.Scatter(x=all_dates, y=curve, name=sym,
                                     line=dict(color=PALETTE[i % len(PALETTE)], width=2)))
    if ihsg_curve:
        fig_cmp.add_trace(go.Scatter(x=all_dates, y=ihsg_curve, name="IHSG B&H",
                                     line=dict(color="#95a5a6", width=2, dash="dash")))
    fig_cmp.add_hline(y=100, line_dash="dot", line_color="#bdc3c7", line_width=1)
    fig_cmp.update_layout(
        title="🔮 HMM Strategy — Multi-Stock vs IHSG (Base = 100)",
        height=480, template="plotly_white",
        legend=dict(orientation="h", y=1.08),
        hovermode="x unified",
        yaxis_title="Normalised Value",
    )
    fig_cmp.show()


=== BBCA ===
Running walk-forward HMM (120-day window) on 1393 bars ...
Done. 220 trades executed.

=== BBRI ===
Running walk-forward HMM (120-day window) on 1393 bars ...
Done. 180 trades executed.

=== TLKM ===
Running walk-forward HMM (120-day window) on 1393 bars ...
Done. 181 trades executed.

=== ASII ===
Running walk-forward HMM (120-day window) on 1393 bars ...
Done. 230 trades executed.

=== ADRO ===
Running walk-forward HMM (120-day window) on 1393 bars ...
Done. 172 trades executed.

===== Comparison Summary =====


,Symbol,Total Ret,CAGR,Sharpe,Max DD,Win Rate,Trades
0,BBCA,+620.57%,+49.53%,2.94,-5.20%,16.2%,220
1,BBRI,+1119.15%,+66.44%,3.07,-7.12%,20.0%,180
2,TLKM,+1468.18%,+75.20%,3.12,-5.59%,21.0%,181
3,ASII,+938.55%,+61.09%,2.85,-7.71%,15.9%,230
4,ADRO,+4076.71%,+113.89%,2.79,-28.35%,15.9%,172


---
## Cell 4 — Live Signal (Current)

In [30]:
# ── Cell 4 · Live signal ──────────────────────────────────────────────────────
LIVE_SYMBOL   = "AKRA"
LIVE_LOOKBACK = 300   # days

r = analyze_stock(LIVE_SYMBOL, LIVE_LOOKBACK)
show_signal_card(r)
show_regime_table(r)

Regime,Days,%,Avg Ret/day,Volatility,Persistence,Ann.Sharpe
Bear 📉,70,24.9%,-1.486%,2.95%,77.1%,-7.99
Sideways ➡️,162,57.7%,-0.029%,1.70%,79.5%,-0.27
Bull 📈,49,17.4%,2.577%,4.32%,53.1%,9.46


---
## Cell 5 — Universe Batch Scanner

In [21]:
# ── Cell 5 · Universe scanner ─────────────────────────────────────────────────
def scan_universe(symbols, lookback=200):
    rows = []
    for sym in symbols:
        try:
            r  = analyze_stock(sym, lookback)
            cs = r["current_stats"]
            rows.append(dict(
                Symbol=r["symbol"], Name=r["name"],
                Signal=r["signal"], Regime=r["regime"],
                Confidence=f"{r['confidence']:.1f}%",
                Strength=r["strength"],
                AvgRet=f"{cs.get('avg_return', 0):.3f}%",
                Volatility=f"{cs.get('volatility', 0):.2f}%",
                Price=f"Rp {float(r['data']['Close'].iloc[-1]):,.0f}",
            ))
            print(f"  {sym:8s} -> {r['signal']:4s} ({r['confidence']:.0f}%)")
        except Exception as e:
            print(f"  ERROR {sym}: {e}")
    if not rows:
        return pd.DataFrame()
    df = pd.DataFrame(rows)
    order = {"BUY": 0, "HOLD": 1, "SELL": 2}
    df["_o"] = df["Signal"].map(order)
    df = (df.sort_values(["_o", "Strength"], ascending=[True, False])
            .drop(columns="_o").reset_index(drop=True))
    return df

SCAN_LIST = ["TUGU", "TSPC", "TAPG", "BTPS", "KLBF", "HEXA", "KMDS", "LSIP", "PGAS", "RALS", "MCOL"]
print(f"Scanning {len(SCAN_LIST)} stocks ...")
summary = scan_universe(SCAN_LIST)
print()

def clr(v):
    return {"BUY":  "background-color:#d5f5e3;color:#1e8449",
            "SELL": "background-color:#fadbd8;color:#922b21",
            "HOLD": "background-color:#fef9e7;color:#7d6608"}.get(v, "")

display(summary.style.applymap(clr, subset=["Signal"]))

Scanning 11 stocks ...
  TUGU     -> BUY  (100%)
  TSPC     -> HOLD (98%)
  TAPG     -> SELL (90%)
  BTPS     -> SELL (100%)
  KLBF     -> HOLD (98%)
  HEXA     -> BUY  (100%)
  KMDS     -> BUY  (100%)
  LSIP     -> BUY  (98%)
  PGAS     -> HOLD (100%)
  RALS     -> HOLD (66%)
  MCOL     -> SELL (99%)



,Symbol,Name,Signal,Regime,Confidence,Strength,AvgRet,Volatility,Price
0,TUGU,,BUY,Bull 📈,100.0%,10,1.268%,3.64%,"Rp 1,440"
1,HEXA,,BUY,Bull 📈,100.0%,10,0.255%,0.65%,"Rp 4,530"
2,KMDS,,BUY,Bull 📈,100.0%,10,0.600%,3.27%,Rp 635
3,LSIP,,BUY,Bull 📈,98.2%,10,0.762%,2.35%,"Rp 1,195"
4,TSPC,,HOLD,Sideways ➡️,97.9%,10,-0.170%,0.89%,"Rp 2,760"
5,KLBF,Kalbe Farma,HOLD,Sideways ➡️,98.0%,10,-0.124%,2.17%,"Rp 1,080"
6,PGAS,Perusahaan Gas Negara,HOLD,Sideways ➡️,99.8%,10,0.006%,1.10%,"Rp 2,180"
7,RALS,,HOLD,Sideways ➡️,65.9%,7,0.051%,1.12%,Rp 500
8,TAPG,,SELL,Bear 📉,89.9%,10,-0.441%,3.11%,"Rp 1,470"
9,BTPS,,SELL,Bear 📉,100.0%,10,-1.321%,2.38%,"Rp 1,190"


---
## 📌 Quick Reference

| Variable | Recommended range |
|---|---|
| `TRAIN_WINDOW` | 100 – 150 days |
| `CONFIDENCE_THRESHOLD` | 0.60 – 0.75 |
| `STOP_LOSS_PCT` | 0.05 – 0.10 |
| `TRANSACTION_COST` | 0.0015 (IDX avg) |
| `N_REGIMES` | 3 (try 4 for finer granularity) |

> ⚠️ **Disclaimer:** Research tool only — not financial advice. Always manage your own risk.